# Construirea Aplicațiilor de Generare a Imaginilor (OpenAI)

Există mai mult în LLM-uri decât generarea de text. Poți genera și imagini din descrieri textuale. Imaginile ca modalitate sunt utile în domenii precum MedTech, arhitectură, turism, dezvoltare de jocuri, marketing și altele. În această lecție lucrăm cu modelele **GPT Image** de astăzi de pe platforma OpenAI.

## Obiective de învățare

La sfârșitul acestei lecții vei putea:

- Explica ce este generarea de imagini și unde este utilă.
- Înțelege familia de modele `gpt-image` și cum se diferențiază de modelele vechi DALL·E.
- Construiește o aplicație de generare a imaginilor și editează imagini.

## Ce este generarea de imagini?

Modelele de generare a imaginilor creează imagini pornind de la un prompt text. Modelele moderne precum `gpt-image` învață relația dintre text și imagini în timpul antrenamentului, apoi transformă treptat zgomotul aleatoriu într-o imagine care se potrivește promptului tău.

Două familii cunoscute de modele de imagini sunt:

- **`gpt-image` (OpenAI)** — generația curentă folosită în această lecție. Suportă generarea de imagini din text și editarea imaginilor (înpaint cu o mască).
- **Midjourney** — un model popular terț, cu propriul serviciu și flux de lucru bazat pe Discord.

> Modelele OpenAI mai vechi de imagini — **DALL·E 2** și **DALL·E 3** — sunt modele vechi. DALL·E 3 nu mai este disponibil pentru noile implementări, iar funcționalități precum `create_variation` existau doar în DALL·E 2. Folosește modelele `gpt-image` pentru aplicațiile noi.

> **Important:** modelele `gpt-image` returnează imaginea generată ca **base64** (`b64_json`), nu ca URL. Codul tău decodează șirul base64 în bytes și îl salvează — nu există un URL de imagine pentru descărcare.


## Construirea primei tale aplicații de generare a imaginilor

Deci, ce este necesar pentru a construi o aplicație de generare a imaginilor? Ai nevoie de următoarele biblioteci:

- **python-dotenv**, este recomandat să folosești această bibliotecă pentru a păstra secretele într-un fișier *.env* separat de cod.
- **openai**, această bibliotecă este cea pe care o vei folosi pentru a interacționa cu API-ul OpenAI.
- **pillow**, pentru a lucra cu imagini în Python.


1. Creează un fișier *.env* cu următorul conținut:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Colectați bibliotecile de mai sus într-un fișier numit *requirements.txt* astfel:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Apoi, creați un mediu virtual și instalați bibliotecile:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Pentru Windows, folosește următoarele comenzi pentru a crea și a activa mediul tău virtual:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Adaugă următorul cod în fișierul numit *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Creează obiectul OpenAI (citește OPENAI_API_KEY din fișierul tău .env)
    client = openai.OpenAI()


    try:
        # Creează o imagine folosind API-ul de generare a imaginilor
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Introdu textul promptului aici
            size='1024x1024',
            n=1
        )
        # Setează directorul pentru imaginea stocată
        image_dir = os.path.join(os.curdir, 'images')

        # Dacă directorul nu există, creează-l
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Inițializează calea imaginii (reține că tipul de fișier trebuie să fie png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # Modelele gpt-image returnează imaginea ca base64 (b64_json), nu un URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Afișează imaginea în vizualizatorul implicit de imagini
        image = Image.open(image_path)
        image.show()

    # prinde excepțiile
    except openai.BadRequestError as err:
        print(err)

    ```

Hai să explicăm acest cod:

- Mai întâi, importăm bibliotecile de care avem nevoie, inclusiv biblioteca OpenAI, biblioteca dotenv, modulul base64 și biblioteca Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- După aceea, creăm clientul, care citește cheia API din fișierul tău ``.env``.

    ```python
    # Creează obiectul OpenAI
    client = openai.OpenAI()
    ```

- Următorul pas este să generăm imaginea:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Introduceți textul promptului aici
        size='1024x1024',
        n=1
    )
    ```

    Modelele `gpt-image` returnează imaginea ca un șir **base64** în `data[0].b64_json`. Îl decodăm în octeți și îl scriem într-un fișier — nu există un URL pentru descărcare.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- În cele din urmă, deschidem imaginea și folosim vizualizatorul standard de imagini pentru a o afișa:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Mai multe detalii despre generarea imaginii

Să analizăm parametrii funcției `images.generate`:

- **model**, este modelul de imagine de folosit (de exemplu `gpt-image-1`).
- **prompt**, este textul folosit pentru a genera imaginea. Aici este "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, este dimensiunea imaginii generate (de exemplu `1024x1024`, `1536x1024`, `1024x1536`, sau `"auto"`).
- **n**, este numărul de imagini generate. Aici generăm una.

> Modelele de imagine nu iau un parametru `temperature` — acesta este un control pentru generarea de text. Pentru a obține varietate, apelează API-ul din nou; pentru a reduce varietatea, fă promptul mai specific.

## Capabilități suplimentare ale generării de imagini

Ai văzut cum să generezi o imagine cu câteva linii de Python. Modelele `gpt-image` pot de asemenea să **editeze** o imagine existentă. Furnizând o imagine, o **mască** opțională (care marchează zona de schimbat) și un prompt, poți modifica o parte a imaginii — de exemplu, să adaugi o pălărie iepurașului nostru.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# editările sunt returnate și în format base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Imaginea de bază conține numai iepurele; imaginea finală are pălăria pe iepure.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
